# phase_2 LLM branch — finetune the scene-graph extractor (Qwen2.5-7B QLoRA)

Step 6: 4-bit QLoRA finetune of `Qwen/Qwen2.5-7B-Instruct` to turn **report text + the list of
available regions** into the compact per-region findings JSON (the silver scene-graph target).
Drive-resumable (service account) like the detector notebook.

**Run BEFORE this notebook (local, no GPU):** build the SFT data and upload it.
```bash
python phase_2/build_sft_dataset.py --metadata data/mimic_metadata_final.jsonl \
    --scene-root <chest-imagenome> --out sg_sft --keep-empty-frac 0.1
# -> upload the sg_sft/ folder (train.jsonl + val.jsonl) as a Kaggle dataset named `sg-sft`
```
`sg_vocab.json` (extract_sg_vocab.py) is NOT needed here — only later at step 7 (inference).

**Inputs:** `sg-sft` dataset. Settings → **GPU** (P100/T4 16 GB ok), **Internet On** (HF download +
rclone). Then Run All. 2nd session on: set `RESUME = 1`.

## CONFIG — edit these

In [ ]:
import os
PHASE2_SRC = '/kaggle/working/repo/phase_2'         # cloned from GitHub (next cell)
SG_SFT     = '/kaggle/input/sg-sft'                 # build_sft_dataset.py output (train.jsonl, val.jsonl)

MODEL    = 'Qwen/Qwen2.5-7B-Instruct'
EPOCHS   = 2
RUN_NAME = 'sg_lora'
RESUME   = 0          # 1 from the 2nd session on

# Drive via SERVICE ACCOUNT (no rclone.conf/token); dhint: root = CHEX-DATA
DRIVE_FOLDER_ID = '1c6AJ3fjsL449kiMK4xiXfnzwzA4gIo0O'   # CHEX-DATA folder id
RCLONE_REMOTE   = 'dhint:sg_lora_runs'                  # checkpoints -> CHEX-DATA/sg_lora_runs/<RUN_NAME>

for k, v in dict(PHASE2_SRC=PHASE2_SRC, SG_SFT=SG_SFT, MODEL=MODEL, EPOCHS=EPOCHS,
                 RUN_NAME=RUN_NAME, RESUME=RESUME, DRIVE_FOLDER_ID=DRIVE_FOLDER_ID,
                 RCLONE_REMOTE=RCLONE_REMOTE).items():
    os.environ[k] = str(v)
print('config set | resume', RESUME)

In [ ]:
# --- get the code from GitHub. Internet ON. ---
!rm -rf /kaggle/working/repo && git clone -q https://github.com/hiennguyendang/phase_2_3_4_5 /kaggle/working/repo
!ls /kaggle/working/repo/phase_2 | head

## 1 — install rclone + auth via a Google **service account**
One-time: Kaggle Secret **`GDRIVE_SA`** = full service-account JSON, **share `CHEX-DATA`** (Editor)
with the SA's `client_email`, and paste its folder id into `DRIVE_FOLDER_ID` (CONFIG).

In [ ]:
%%bash
set -e
if ! command -v rclone >/dev/null 2>&1; then
  cd /kaggle/working && curl -sLO https://downloads.rclone.org/rclone-current-linux-amd64.zip
  unzip -q -o rclone-current-linux-amd64.zip && cp rclone-*-linux-amd64/rclone /usr/local/bin/ && chmod +x /usr/local/bin/rclone
fi
rclone version | head -1

In [ ]:
import os, pathlib
# Define 'dhint' Drive remote from a service account + folder id, via env vars (no rclone.conf/token).
SYNC_OK = '0'
try:
    from kaggle_secrets import UserSecretsClient
    sa = UserSecretsClient().get_secret('GDRIVE_SA')
    sa_path = '/kaggle/working/gdrive_sa.json'; pathlib.Path(sa_path).write_text(sa)
    os.environ['RCLONE_CONFIG_DHINT_TYPE'] = 'drive'
    os.environ['RCLONE_CONFIG_DHINT_SERVICE_ACCOUNT_FILE'] = sa_path
    os.environ['RCLONE_CONFIG_DHINT_ROOT_FOLDER_ID'] = os.environ['DRIVE_FOLDER_ID']
    if os.system('rclone mkdir "%s"' % os.environ['RCLONE_REMOTE']) == 0:
        SYNC_OK = '1'; print('remote OK (service account) ->', os.environ['RCLONE_REMOTE'])
    else:
        print('[WARN] remote unreachable -> NO Drive sync. Check folder share + DRIVE_FOLDER_ID')
except Exception as e:
    print('[WARN] GDRIVE_SA secret not set -> NO Drive sync:', e)
os.environ['SYNC_OK'] = SYNC_OK; print('SYNC_OK =', SYNC_OK)

## 2 — copy code + install LLM deps
If the trl API errors (it changes across versions), pin: `trl==0.11.4 transformers==4.44.2`.

In [ ]:
!rm -rf /kaggle/working/phase_2 && cp -r "$PHASE2_SRC" /kaggle/working/phase_2
%cd /kaggle/working/phase_2
!pip install -q "transformers>=4.44" "trl>=0.11,<0.12" "peft>=0.12" "bitsandbytes>=0.43" "accelerate>=0.33" "datasets>=2.20"
!python -c "import json; n=sum(1 for _ in open('$SG_SFT/train.jsonl')); print('train samples:', n)"  # sanity

## 3 — finetune (Drive-synced)
Pushes the run dir to Drive every checkpoint (`--save-steps`). If the session dies, set `RESUME=1`
in CONFIG and Run All again — it pulls the checkpoints from Drive and continues.

In [ ]:
import os
S = ('--sync-remote "%s/%s"' % (os.environ['RCLONE_REMOTE'], os.environ['RUN_NAME'])) if os.environ.get('SYNC_OK') == '1' else ''
R = '--resume' if int(os.environ['RESUME']) else ''
os.environ.update(S_FLAG=S, R_FLAG=R)
print('flags:', S or '(no sync)', R or '(fresh)')
!python finetune_sg_llm.py --data-dir "$SG_SFT" --out /kaggle/working/sg_lora \
  --model "$MODEL" --epochs $EPOCHS $S_FLAG $R_FLAG

## 4 — grab the LoRA adapters
Already on Drive (final push). Pull anywhere: `rclone copy dhint:sg_lora_runs/sg_lora ./sg_lora`.
Or download from the **Output** tab. Step 7 (`build_pseudo_scene_graph.py`) loads these on the base model.

In [ ]:
import os
if os.environ.get('SYNC_OK') == '1':
    os.system('rclone copy /kaggle/working/sg_lora "%s/%s" --quiet' % (os.environ['RCLONE_REMOTE'], os.environ['RUN_NAME']))
    print('adapters pushed -> %s/%s' % (os.environ['RCLONE_REMOTE'], os.environ['RUN_NAME']))
!ls -lh /kaggle/working/sg_lora | head